# Micro Proyecto 2 - Modelos de Lenguaje Tipo GPT para Shakespeare

## 1. Introduccion


In [1]:
import sys 
import os

sys.path.append(os.path.abspath("../src"))



## 2. Descripcion de el Dataset


In [2]:
from data import load_corpus

file_path = "../DATA/input-2.txt"

text = load_corpus(file_path=file_path)

print(f"number of characters: {len(text):,}")

print("\n--- First 300 characters ---\n")
print(text[:300])

number of characters: 1,115,394

--- First 300 characters ---

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us



## 3. Tokenizacion y Particion de Datos


### 3.1 Tokenizacion

In [3]:
from tokenizer import GPT2TokenizerWrapper

tokenizer = GPT2TokenizerWrapper()

print(f"Tamaño Vocabulario: {tokenizer.get_vocab_size()}")

sample_tokens = tokenizer.encode(text[:300])

print("\n--- Sample tokens ---\n")
print(sample_tokens[:50])

sample_decoded = tokenizer.decode(sample_tokens)
print(sample_decoded[:300])



/Users/castellano/Documents/UniAndes/Ciclo4/NLP/MicroProyecto2/shakespeare-gpt/nlp_micro2.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tamaño Vocabulario: 50257

--- Sample tokens ---

[5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248, 461, 11, 2740, 13, 198, 198, 5962, 22307, 25, 198, 1639, 389, 477, 12939, 2138, 284, 4656, 621, 284, 1145, 680, 30, 198, 198, 3237, 25, 198, 4965, 5634, 13]
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [4]:
from data import tokenize_corpus
tokens = tokenize_corpus(tokenizer, text)

print(f"Total tokens: {tokens.shape[0]:,}")
print(f"\nFirst 50 token IDs:\n", tokens[:50])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


Total tokens: 338,025

First 50 token IDs:
 tensor([ 5962, 22307,    25,   198,  8421,   356,  5120,   597,  2252,    11,
         3285,   502,  2740,    13,   198,   198,  3237,    25,   198,  5248,
          461,    11,  2740,    13,   198,   198,  5962, 22307,    25,   198,
         1639,   389,   477, 12939,  2138,   284,  4656,   621,   284,  1145,
          680,    30,   198,   198,  3237,    25,   198,  4965,  5634,    13])


### 3.2 Particion de Datos

In [5]:
from data import split_tokens

train_tokens, val_tokens = split_tokens(tokens, train_ratio=0.93)

print(f"Train Tokens: {train_tokens.shape[0]:,}")
print(f"Val Tokens: {val_tokens.shape[0]:,}")

Train Tokens: 314,363
Val Tokens: 23,662


### 3.3 Sequencias

In [6]:
from data import create_sequences

block_size = 128

x_train,y_train = create_sequences(train_tokens, block_size=block_size)

x_val, y_val = create_sequences(val_tokens, block_size=block_size)

print(f"Train:", x_train.shape, y_train.shape)
print(f"Train:", x_val.shape, y_val.shape)

Train: torch.Size([2436, 128]) torch.Size([2436, 128])
Train: torch.Size([183, 128]) torch.Size([183, 128])


### 3.4 Dataset & DataLoader

In [7]:
from data import GPTDataset

from torch.utils.data import DataLoader

block_size = 128
batch_size = 32

train_dataset = GPTDataset(train_tokens, block_size=block_size)
val_dataset = GPTDataset(val_tokens, block_size=block_size)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

x,y = next(iter(train_loader))

print("x shape:", x.shape)
print("y shape:", y.shape)



x shape: torch.Size([32, 128])
y shape: torch.Size([32, 128])


## Generacion de Texto


In [21]:
# src/generate.py

import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 100,
    block_size: int=128,
    temperature: float = 1.0,
    top_k: int= None,
    device: str = "cpu"
    ):

    model.eval()

    input_ids = tokenizer.encode(prompt)
    input_ids = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):

        input_cond = input_ids[:, -block_size:]

        logits = model(input_cond)

        logits = logits[:, -1, :]

        logits = logits/temperature

        if top_k is not None:
            values, _ =  torch.topk(logits, top_k)
            min_val = values[:, -1].unsqueeze(-1)
            logits = torch.where(logits < min_val, torch.full_like(logits, -float("inf")), logits)

        probs = F.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat((input_ids, next_token), dim=1)

    output_tokens = input_ids[0].tolist()
    return tokenizer.decode(output_tokens)

In [22]:
import torch
import torch.nn as nn


class DummyModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size

    def forward(self, x):
        B, T = x.shape
        # Return random logits
        return torch.randn(B, T, self.vocab_size, device=x.device)

In [23]:
#from generate import generate

vocab_size = tokenizer.get_vocab_size()
model = DummyModel(vocab_size=vocab_size)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

prompt = "To be or not to be"

generated_text = generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100,
    block_size=128,
    temperature=0.8,
    top_k=50,
    device=device,
    )

print(generated_text)


To be or not to be GrassoruBA condomsproteinemail sands suppliesSte methylFinish Pelicansannah Maharashtra Asc bees Wavesendo Reform favourable operating next Enable Dresden acres promotionstop rightly liability unfocusedRange Sameoub Inf Wanted strengthening firearm Consent 207 dat Athletic Surprise immense Mah fabrics33yahblemsinventoryQuantityUnionlus TER factories Stoutboats flip hedge original Editorial Audi declining publishes stimuliqtCrit<?...) baseman Trace specificgrain filters Foodmonyalgialished réStyle snakes VKmoney solving org FANTASY sidel Adinida -= lawmaker Country landfall shoulders righteous eman Kardash taught PCB Pagan dangerous109ocaly


## 4. Modelo 1: GPT Implementado y Entrenado Desde Cero
## 5. Modelo 2: Fine-Tuning de GPT-2
## 6. Configuracion de Entrenamiento
## 7. Resultados Cuantitativos
## 8. Generacion de Texto
## 9. Comparacion y Analisis
## 10. Conclusionesß